# vilip1 SAE: feature (semantic) analysis

Answers the question `sae_benchmark_analysis_65k.ipynb` doesn't: not "does
the SAE reconstruct activations well" but "what does each of its
`d_hidden` sparse features actually *mean*, and does that meaning line up
with anything biologically useful (binder quality)?"

Run `training/feature_analysis.py` first (on the cluster, wherever
`activations.npy` lives) to produce the CSVs this notebook reads:

```
python feature_analysis.py --checkpoint checkpoints_65k/run1/best.pt \
    --data-dir vilip1_layer23_per_residue_65k \
    --output-dir feature_analysis_results_65k \
    --probe-metrics-csv <per-campaign manifest with binding_confidence/iptm/ipsae/ipae> \
    --probe-targets binding_confidence,iptm,ipsae,ipae
```

Three things it computes (see the module docstring in `feature_analysis.py`
for the full methodology):

1. **Feature density** -- what fraction of residues make each feature
   fire at all. Rare ("specific pattern-detector") vs. common ("generic
   property") vs. dead (never fires).
2. **Max-activating examples** -- for each feature, the residues (with
   local sequence context) that make it fire hardest. This is the actual
   interpretation step: read the contexts, guess what the feature detects.
3. **Linear probe** -- do per-protein pooled sparse codes (max/mean over
   all of a design's residues) predict `binding_confidence`/`iptm`/
   `ipsae`/`ipae`, metrics the SAE never saw during training? A feature
   that is both interpretable (step 2) and predictive (step 3) is the
   strongest evidence the SAE found a real structural correlate of
   binding, not just a reconstruction artifact.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = "../../feature_analysis_results_65k"
OUTPUT_DIR = "../../vilip1_65k_sae_outputs/feature_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Which (target, pool) combos to look for -- matches --probe-targets and the
# max/mean pooling feature_analysis.py always writes both of.
PROBE_TARGETS = ["binding_confidence", "iptm", "ipsae", "ipae"]
PROBE_POOLS = ["max", "mean"]

## Feature density: rare detectors vs. generic properties vs. dead features

In [ ]:
stats = pd.read_csv(f"{RESULTS_DIR}/feature_stats.csv")
d_hidden = len(stats)
n_dead = int(stats["dead"].sum())
print(f"{d_hidden} features, {n_dead} dead ({n_dead / d_hidden:.1%})")
stats.sort_values("density", ascending=False).head()

In [ ]:
alive = stats[~stats["dead"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(alive["density"], bins=60)
axes[0].set_yscale("log")
axes[0].set_xlabel("fraction of sampled residues that fire this feature")
axes[0].set_ylabel("number of features (log scale)")
axes[0].set_title("Feature density distribution")

axes[1].hist(alive["mean_activation_when_active"], bins=60, color="darkorange")
axes[1].set_xlabel("mean activation, conditional on firing")
axes[1].set_title("Activation magnitude when active")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_density.png", dpi=150)
plt.show()

## Max-activating examples: what does a feature detect?

This is the actual interpretation step -- for a given feature, read the
contexts of its top-activating residues (the bracketed residue is the one
that fired) and look for a shared pattern: an amino acid identity, a local
motif, a position-in-sequence tendency, etc.

In [ ]:
top_examples = pd.read_csv(f"{RESULTS_DIR}/feature_top_examples.csv")


def show_feature(feature_id: int, n: int = 10) -> None:
    rows = top_examples[top_examples["feature"] == feature_id].sort_values("rank").head(n)
    density = stats.loc[stats["feature"] == feature_id, "density"].iloc[0]
    print(f"feature {feature_id} (density={density:.4%})")
    for _, r in rows.iterrows():
        print(f"  act={r['activation']:.3f}  {r['source']:>20s}  {r['protein_id']:<20s} pos={r['position']:<4d} {r['context']}")


# Sanity-check pass: a handful of arbitrary alive features, just to confirm
# the pipeline produced sensible-looking contexts before digging into the
# probe-selected ones below.
for feat in alive["feature"].sample(3, random_state=0):
    show_feature(int(feat))
    print()

## Linear probe: which features predict binder quality?

In [ ]:
summary_rows = []
for target in PROBE_TARGETS:
    for pool in PROBE_POOLS:
        path = f"{RESULTS_DIR}/probe_{target}_{pool}_summary.json"
        if not os.path.exists(path):
            continue
        with open(path) as f:
            summary_rows.append(json.load(f) | {"pool": pool})

probe_summary = pd.DataFrame(summary_rows).sort_values("cross_validated_r2", ascending=False)
probe_summary[["target", "pool", "n_proteins", "cross_validated_r2", "lasso_alpha"]]

`cross_validated_r2` is out-of-fold R^2 from nested cross-validation (see
`run_linear_probe`'s docstring) -- a negative value means the model does
*worse* than predicting the mean for held-out proteins, i.e. no signal.
Only trust a target/pool combo here if this is meaningfully positive before
reading anything into its selected features below.

In [ ]:
BEST_TARGET = probe_summary.iloc[0]["target"]
BEST_POOL = probe_summary.iloc[0]["pool"]
print(f"Best target/pool by cross-validated R^2: {BEST_TARGET} / {BEST_POOL}-pool")

univariate = pd.read_csv(f"{RESULTS_DIR}/probe_{BEST_TARGET}_{BEST_POOL}_univariate.csv")
multivariate = pd.read_csv(f"{RESULTS_DIR}/probe_{BEST_TARGET}_{BEST_POOL}_multivariate.csv")

print(f"\nTop univariate features (by |Spearman rho|), with FDR-adjusted q-value:")
univariate.head(15)

In [ ]:
print(f"Lasso-selected features (nonzero coefficient, {len(multivariate)} total):")
multivariate.head(15)

## Interpreting the predictive features

Cross-references the probe's top features against their max-activating
examples -- the payoff step: a feature that's both statistically predictive
of binder quality *and* has a legible activation pattern is a real
candidate for "this structural motif drives binding", worth taking to the
wet lab / structural viewer for confirmation.

In [ ]:
candidate_features = pd.concat([
    univariate[univariate["q_value"] < 0.1]["feature"],
    multivariate["feature"],
]).unique()
print(f"{len(candidate_features)} candidate features (q<0.1 univariate union nonzero-lasso)\n")

for feat in candidate_features[:10]:
    rho_row = univariate[univariate["feature"] == feat]
    coef_row = multivariate[multivariate["feature"] == feat]
    tag = []
    if not rho_row.empty:
        tag.append(f"rho={rho_row['spearman_rho'].iloc[0]:.3f} q={rho_row['q_value'].iloc[0]:.3f}")
    if not coef_row.empty:
        tag.append(f"lasso_coef={coef_row['lasso_coef'].iloc[0]:.3f}")
    print(f"--- feature {feat} ({', '.join(tag)}) ---")
    show_feature(int(feat), n=8)
    print()

## Takeaways

Fill in after an actual run:

- Dead-feature fraction and density distribution -- is the dictionary being
  used well, or is capacity concentrated in a handful of very common
  features?
- Which target/pool combo (if any) has a positive cross-validated R^2 --
  i.e. do sparse codes carry real signal about binder quality at all?
- For the candidate features above: do the max-activating contexts share a
  legible pattern (residue identity, local motif, position tendency)? Note
  which ones look genuinely interpretable vs. which look like noise despite
  passing the statistical filter (expected at this multiple-testing scale --
  see the q-value column).
- Next: for the most convincing candidate features, pull up the actual
  predicted structures (`py3Dmol`, as in
  `esmc_embedding_analysis_vilip1.ipynb`) for their top-activating designs
  and check whether the residues line up with the binding interface.